In [1]:
# ===============================================================
# 1. Imports and Setup
# ===============================================================
import os, json, re, time
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt

# Paths aligned with your pipeline.yaml
DIR_DOC_TOPICS = "outputs_doc_topics"
DIR_CHUNKS = "outputs_chunks"
os.makedirs("outputs_metrics", exist_ok=True)

# Embedding model (same as Contextizer)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Comparison config
TOP_K = 40  # keywords per method for fairness


/Users/martinjurado/Desktop/prjs/T2G/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===============================================================
# 2. Helper Functions
# ===============================================================

def list_json(dirpath):
    return [os.path.join(dirpath, f) for f in os.listdir(dirpath) if f.endswith(".json")] if os.path.exists(dirpath) else []

def infer_doc_id_from_filename(fname):
    m = re.match(r"^(DOC-[A-Za-z0-9\-]+)", fname)
    return m.group(1) if m else os.path.splitext(fname)[0]

def load_hybrid_keywords_from_doc_topics(path):
    """Extract hybrid Contextizer keywords from meta.topics_doc.keywords_global or topics[0].keywords."""
    with open(path, "r") as f:
        d = json.load(f)
    meta = d.get("meta", {})
    td = meta.get("topics_doc", {}) or {}
    kws = td.get("keywords_global", [])
    if not kws:
        topics = td.get("topics", [])
        if topics and isinstance(topics, list) and "keywords" in topics[0]:
            kws = topics[0]["keywords"]
    return [k.strip().lower() for k in kws if isinstance(k, str) and k.strip()]

def load_texts(doc_topics_path=None, chunks_path=None):
    """Prefer texts from doc_topics (pages[].blocks[].text_norm), fallback to chunks[].text."""
    texts = []
    if doc_topics_path and os.path.exists(doc_topics_path):
        with open(doc_topics_path, "r") as f:
            d = json.load(f)
        for page in d.get("pages", []):
            for b in page.get("blocks", []):
                t = b.get("text_norm") or b.get("text_clean") or b.get("text")
                if t and isinstance(t, str) and t.strip():
                    texts.append(t.strip())
    if not texts and chunks_path and os.path.exists(chunks_path):
        with open(chunks_path, "r") as f:
            d = json.load(f)
        for ch in d.get("chunks", []):
            t = ch.get("text")
            if t and isinstance(t, str) and t.strip():
                texts.append(t.strip())
    return texts


In [3]:
# ===============================================================
# 3. Metric Utilities
# ===============================================================

def cosine_coherence(keywords, model):
    ks = [k for k in keywords if isinstance(k, str) and k.strip()]
    if len(ks) < 2: return 0.0
    emb = model.encode(ks, normalize_embeddings=True)
    sim = cosine_similarity(emb)
    np.fill_diagonal(sim, np.nan)
    return float(np.nanmean(sim))

def lexical_diversity(keywords):
    ks = [k for k in keywords if isinstance(k, str) and k.strip()]
    return (len(set(ks)) / len(ks)) if ks else 0.0

def jaccard_overlap(a, b):
    A, B = set(a), set(b)
    return (len(A & B) / len(A | B)) if (A and B) else 0.0

def coverage_at_k(a, b, k=10):
    A, B = set(a[:k]), set(b[:k])
    return len(A & B) / k if k else 0.0


In [4]:
# ===============================================================
# 3B. Extended Semantic Metrics (replaces simple diversity/runtime)
# ===============================================================
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def jaccard_overlap(a, b):
    A, B = set(a), set(b)
    return (len(A & B) / len(A | B)) if (A and B) else 0.0

def semantic_diversity(keywords, model):
    ks = [k for k in keywords if isinstance(k, str) and k.strip()]
    if len(ks) < 2:
        return 0.0
    emb = model.encode(ks, normalize_embeddings=True)
    sim = cosine_similarity(emb)
    n = sim.shape[0]
    tri = sim[np.triu_indices(n, k=1)]
    return float(1.0 - np.nanmean(tri)) if tri.size > 0 else 0.0

def distinct_n_ratio(keywords, n=2):
    toks = [t for k in keywords for t in k.lower().split()]
    if len(toks) < n:
        return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    return len(set(ngrams)) / max(1, len(ngrams))


In [5]:
# ===============================================================
# 4. BERTopic Runner (deterministic, stopwords cleaned)
# ===============================================================

def run_bertopic(texts, model, top_n_words=10):
    """Robust BERTopic runner: handles small or homogeneous corpora gracefully."""
    # Validate input
    texts = [t.strip() for t in texts if isinstance(t, str) and t.strip()]
    if len(texts) < 2:
        # Fallback: single text → TF-IDF extraction
        from sklearn.feature_extraction.text import TfidfVectorizer
        tfidf = TfidfVectorizer(max_features=top_n_words, stop_words="english")
        tfidf.fit(texts)
        return list(tfidf.get_feature_names_out()), pd.DataFrame({"Topic": [0], "Name": ["fallback"]})

    # Build BERTopic
    from umap import UMAP
    from hdbscan import HDBSCAN
    from sklearn.feature_extraction.text import CountVectorizer
    import re

    stoplist = [
        "the", "to", "and", "of", "in", "a", "is", "for", "on", "that", "with", "as", "at",
        "by", "an", "be", "are", "or", "this", "from", "it", "not", "was", "will",
        "de", "la", "el", "en", "los", "las", "por", "un", "una", "y", "que", "del", "se", "al"
    ]

    umap_model = UMAP(n_neighbors=min(10, len(texts)-1), n_components=5, min_dist=0.0, random_state=42)
    hdbscan_model = HDBSCAN(min_cluster_size=2, metric='euclidean',
                            cluster_selection_method='eom', prediction_data=True)
    vectorizer_model = CountVectorizer(
        stop_words=stoplist,
        lowercase=True,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]{2,}\b",
        max_df=1.0,
        min_df=1
    )

    try:
        tm = BERTopic(
            embedding_model=model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            top_n_words=top_n_words,
            calculate_probabilities=True,
            verbose=False
        )
        topics, probs = tm.fit_transform(texts)
        info = tm.get_topic_info()

        # keep valid topics only
        if "Topic" not in info.columns or len(info) == 0:
            raise ValueError("Empty topic info")

        # build keyword list
        kw = []
        for name in info["Name"].dropna().values:
            words = [
                w.strip().lower()
                for w in str(name).split(",")
                if len(w.strip()) > 2 and re.match(r"^[a-zA-Z\-]+$", w.strip())
            ]
            kw.extend(words[:top_n_words])
        if not kw:
            raise ValueError("No keywords extracted")
        return kw, info

    except Exception as e:
        # 3️ graceful fallback
        from sklearn.feature_extraction.text import TfidfVectorizer
        tfidf = TfidfVectorizer(max_features=top_n_words, stop_words=stoplist)
        tfidf.fit(texts)
        fallback_kw = list(tfidf.get_feature_names_out())
        return fallback_kw, pd.DataFrame({"Topic": [0], "Name": [f"fallback ({str(e)[:30]})"]})




In [6]:
# ===============================================================
# 5. Pair documents (doc_topics ↔ chunks)
# ===============================================================
doc_topics_files = list_json(DIR_DOC_TOPICS)
chunks_files = list_json(DIR_CHUNKS)

doc_topics_map = {infer_doc_id_from_filename(os.path.basename(p)): p for p in doc_topics_files}
chunks_map     = {infer_doc_id_from_filename(os.path.basename(p)): p for p in chunks_files}
doc_ids = sorted(set(doc_topics_map.keys()) | set(chunks_map.keys()))

print(f"✅ {len(doc_ids)} document(s) detected for comparison.")
if doc_ids: print("Examples:", doc_ids[:5])


✅ 6 document(s) detected for comparison.
Examples: ['DOC-0D478638-41BF', 'DOC-1753CAA8-EECC', 'DOC-4B43DEFE-1F7E', 'DOC-89315DD0-7219', 'DOC-D2425F06-5414']


In [7]:
# ===============================================================
# 6. Run Comparison per Document (Enhanced – No Runtime Metric)
# ===============================================================
results = []

for doc_id in tqdm(doc_ids):
    dt_path = doc_topics_map.get(doc_id)
    ch_path = chunks_map.get(doc_id)

    texts = load_texts(dt_path, ch_path)
    if not texts:
        print(f"⚠️ Skipping {doc_id}: no text found.")
        continue

    hybrid_keywords = load_hybrid_keywords_from_doc_topics(dt_path) if dt_path else []
    if not hybrid_keywords:
        print(f"⚠️ Skipping {doc_id}: no hybrid keywords.")
        continue

    # --- Normalize keyword lists ---
    hybrid_k = hybrid_keywords[:TOP_K]

    try:
        bertopic_k, topic_info = run_bertopic(texts, embedder, top_n_words=TOP_K)
    except Exception as e:
        print(f"❌ BERTopic failed for {doc_id}: {e}")
        continue

    # ===============================================================
    #  Compute Metrics (semantic, lexical, and overlap-based)
    # ===============================================================
    coh_h = cosine_coherence(hybrid_k, embedder)
    coh_b = cosine_coherence(bertopic_k, embedder)

    # Semantic diversity (1 - cosine similarity)
    semdiv_h = semantic_diversity(hybrid_k, embedder)
    semdiv_b = semantic_diversity(bertopic_k, embedder)

    # Distinct-2 ratio (lexical diversity)
    d2_h = distinct_n_ratio(hybrid_k, n=2)
    d2_b = distinct_n_ratio(bertopic_k, n=2)

    # Conceptual overlap and coverage
    ovlp = jaccard_overlap(hybrid_k, bertopic_k)
    cov_h_in_b = coverage_at_k(hybrid_k, bertopic_k, k=TOP_K)
    cov_b_in_h = coverage_at_k(bertopic_k, hybrid_k, k=TOP_K)

    # ===============================================================
    #  Append all results
    # ===============================================================
    results.append({
        "doc_id": doc_id,
        "n_texts": len(texts),

        # Coherence
        "coherence_hybrid": coh_h,
        "coherence_bertopic": coh_b,

        # Semantic diversity
        "semantic_diversity_hybrid": semdiv_h,
        "semantic_diversity_bertopic": semdiv_b,

        # Distinct-2 lexical diversity
        "distinct2_hybrid": d2_h,
        "distinct2_bertopic": d2_b,

        # Overlap and coverage
        "overlap": ovlp,
        "coverage_h_in_b": cov_h_in_b,
        "coverage_b_in_h": cov_b_in_h,

        # Keyword sets for qualitative inspection
        "keywords_hybrid": ", ".join(hybrid_k),
        "keywords_bertopic": ", ".join(bertopic_k)
    })

# ===============================================================
#  Save and Display Results
# ===============================================================
if not results:
    raise RuntimeError("❌ No results generated. Check JSON structure and keyword fields.")

df = pd.DataFrame(results)
os.makedirs("outputs_metrics", exist_ok=True)
df.to_csv("outputs_metrics/contextizer_vs_bertopic_metrics.csv", index=False)
print(f"\n✅ Saved → outputs_metrics/contextizer_vs_bertopic_metrics.csv ({len(df)} documents processed)")

display(df.head(10))


100%|████████████████████████████████████████████████████████████████████| 6/6 [00:17<00:00,  2.83s/it]


✅ Saved → outputs_metrics/contextizer_vs_bertopic_metrics.csv (6 documents processed)


,doc_id,n_texts,coherence_hybrid,coherence_bertopic,semantic_diversity_hybrid,semantic_diversity_bertopic,distinct2_hybrid,distinct2_bertopic,overlap,coverage_h_in_b,coverage_b_in_h,keywords_hybrid,keywords_bertopic
0,DOC-0D478638-41BF,2,0.280266,0.265951,0.719734,0.734049,1.0,1.0,0.312500,0.250,0.250,"chicken, sandwich, love, place, coming, ages, ...","abstract, ages, always, any, art, atmosphere, ..."
1,DOC-1753CAA8-EECC,28,0.262833,0.263598,0.737167,0.736402,1.0,1.0,0.219512,0.225,0.225,"gold, bloomberg, year, investors, rally, tuesd...","after, also, bank, been, bloomberg, day, debas..."
2,DOC-4B43DEFE-1F7E,1,0.280141,0.276805,0.719859,0.723195,1.0,1.0,0.375000,0.225,0.225,"sushi, absolutely, best, valley, hands, down, ...","75, absolutely, amazing, best, cooked, extreme..."
3,DOC-89315DD0-7219,1,0.250334,0.268650,0.749666,0.731350,1.0,1.0,0.346154,0.225,0.225,"sandwich, atm, wonderful, vietnamese, shoppe, ...","accepted, atm, baguettes, baked, best, bring, ..."
4,DOC-D2425F06-5414,8,0.313800,0.239507,0.686200,0.760493,1.0,1.0,0.250000,0.250,0.250,"gold, inc, president, market, investors, year,...","analysts, before, corp, gold, has, inc, invest..."
5,DOC-F2357F8F-C829,2,0.253896,0.253943,0.746104,0.746057,1.0,1.0,0.190476,0.200,0.200,"sidebar, year, valley, crowded, out, buzzcatio...","accomodating, buzzcation, crew, crowded, just,..."


In [8]:
# ===============================================================
# 7. Comparative Summary (without runtime)
# ===============================================================
summary = pd.DataFrame({
    "Metric": [
        "Mean Coherence",
        "Semantic Diversity (1 - cos)",
        "Distinct-2 Ratio",
        "Jaccard Overlap",
        "Coverage@K (H→B)",
        "Coverage@K (B→H)"
    ],
    "HybridContextizer": [
        df["coherence_hybrid"].mean(),
        df["semantic_diversity_hybrid"].mean(),
        df["distinct2_hybrid"].mean(),
        df["overlap"].mean(),
        df["coverage_h_in_b"].mean(),
        df["coverage_b_in_h"].mean()
    ],
    "BERTopic": [
        df["coherence_bertopic"].mean(),
        df["semantic_diversity_bertopic"].mean(),
        df["distinct2_bertopic"].mean(),
        df["overlap"].mean(),
        df["coverage_h_in_b"].mean(),
        df["coverage_b_in_h"].mean()
    ]
}).round(4)

print("\n🔍 Comparative Summary (HybridContextizer vs BERTopic):")
display(summary)



🔍 Comparative Summary (HybridContextizer vs BERTopic):


,Metric,HybridContextizer,BERTopic
0,Mean Coherence,0.2735,0.2614
1,Semantic Diversity (1 - cos),0.7265,0.7386
2,Distinct-2 Ratio,1.0000,1.0000
3,Jaccard Overlap,0.2823,0.2823
4,Coverage@K (H→B),0.2292,0.2292
5,Coverage@K (B→H),0.2292,0.2292


In [9]:
# ===============================================================
# 8. Keyword Comparison per Document (for IEEE Table – Updated)
# ===============================================================
cols = [
    "doc_id",
    "coherence_hybrid", "coherence_bertopic",
    "semantic_diversity_hybrid", "semantic_diversity_bertopic",
    "distinct2_hybrid", "distinct2_bertopic",
    "overlap", "coverage_h_in_b", "coverage_b_in_h",
    "keywords_hybrid", "keywords_bertopic"
]

display(df[cols].head(10))

print("\n📊 Example interpretation (Results section):\n")
for _, row in df.iterrows():
    print(f"Document: {row.doc_id}")
    print(f"  - HybridContextizer keywords: {row.keywords_hybrid}")
    print(f"  - BERTopic keywords: {row.keywords_bertopic}")
    print(f"  - Coherence → Hybrid={row.coherence_hybrid:.3f}, BERTopic={row.coherence_bertopic:.3f}")
    print(f"  - Semantic Diversity → Hybrid={row.semantic_diversity_hybrid:.3f}, BERTopic={row.semantic_diversity_bertopic:.3f}")
    print(f"  - Distinct-2 → Hybrid={row.distinct2_hybrid:.3f}, BERTopic={row.distinct2_bertopic:.3f}")
    print(f"  - Overlap={row.overlap:.2f} | Coverage H→B={row.coverage_h_in_b:.2f} | B→H={row.coverage_b_in_h:.2f}\n")


,doc_id,coherence_hybrid,coherence_bertopic,semantic_diversity_hybrid,semantic_diversity_bertopic,distinct2_hybrid,distinct2_bertopic,overlap,coverage_h_in_b,coverage_b_in_h,keywords_hybrid,keywords_bertopic
0,DOC-0D478638-41BF,0.280266,0.265951,0.719734,0.734049,1.0,1.0,0.312500,0.250,0.250,"chicken, sandwich, love, place, coming, ages, ...","abstract, ages, always, any, art, atmosphere, ..."
1,DOC-1753CAA8-EECC,0.262833,0.263598,0.737167,0.736402,1.0,1.0,0.219512,0.225,0.225,"gold, bloomberg, year, investors, rally, tuesd...","after, also, bank, been, bloomberg, day, debas..."
2,DOC-4B43DEFE-1F7E,0.280141,0.276805,0.719859,0.723195,1.0,1.0,0.375000,0.225,0.225,"sushi, absolutely, best, valley, hands, down, ...","75, absolutely, amazing, best, cooked, extreme..."
3,DOC-89315DD0-7219,0.250334,0.268650,0.749666,0.731350,1.0,1.0,0.346154,0.225,0.225,"sandwich, atm, wonderful, vietnamese, shoppe, ...","accepted, atm, baguettes, baked, best, bring, ..."
4,DOC-D2425F06-5414,0.313800,0.239507,0.686200,0.760493,1.0,1.0,0.250000,0.250,0.250,"gold, inc, president, market, investors, year,...","analysts, before, corp, gold, has, inc, invest..."
5,DOC-F2357F8F-C829,0.253896,0.253943,0.746104,0.746057,1.0,1.0,0.190476,0.200,0.200,"sidebar, year, valley, crowded, out, buzzcatio...","accomodating, buzzcation, crew, crowded, just,..."



📊 Example interpretation (Results section):

Document: DOC-0D478638-41BF
  - HybridContextizer keywords: chicken, sandwich, love, place, coming, ages, favorites, elsa, burgers, dragon
  - BERTopic keywords: abstract, ages, always, any, art, atmosphere, been, burgers, but, chicken, china, coming, cool, display, dragon, elsa, favorites, fun, have, here, hot, little, love, my, pepper, place, sandwich, their, they, totally, very, wings
  - Coherence → Hybrid=0.280, BERTopic=0.266
  - Semantic Diversity → Hybrid=0.720, BERTopic=0.734
  - Distinct-2 → Hybrid=1.000, BERTopic=1.000
  - Overlap=0.31 | Coverage H→B=0.25 | B→H=0.25

Document: DOC-1753CAA8-EECC
  - HybridContextizer keywords: gold, bloomberg, year, investors, rally, tuesday, said, losses, going, not
  - BERTopic keywords: after, also, bank, been, bloomberg, day, debasement, debt, get, going, gold, had, has, have, higher, investors, its, losses, make, metals, much, ounce, plan, precious, rally, rates, read, recent, retail, said, s